# I01a — Search Algorithms: BFS, DFS, A*, and MCTS

**COMPSCI 713 — Weeks 8–9: Search & Decision Making**

---

## Overview

Search is one of the oldest and most fundamental problems in AI. From finding a path on a map to playing chess to AlphaGo — all involve searching through a space of possible states to find a goal.

In this lesson you will:
- Implement uninformed search: BFS and DFS
- Implement informed search: Greedy Best-First and A*
- Understand heuristics and admissibility
- Implement adversarial search: Minimax with alpha-beta pruning
- Implement Monte Carlo Tree Search (MCTS) — the engine behind AlphaGo

### COMPSCI 713 Alignment
- **Week 8 Thursday:** Search Algorithms I (Tree search, Uninformed, Informed, A*)
- **Week 9 Monday:** Search Algorithms II (Adversarial search, MCTS)

In [ ]:
import heapq
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque, defaultdict

## 1. Problem Representation

All search problems share the same structure:
- **State:** a description of the world at a point in time
- **Initial state:** where we start
- **Goal test:** is this state the goal?
- **Actions:** what moves are available from a state
- **Transition model:** what state results from an action
- **Path cost:** cost of reaching a state

In [ ]:
# ── Grid World ─────────────────────────────────────────────────────────────
# 0 = free, 1 = wall, S = start, G = goal

GRID = [
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 1, 0, 1, 0],
    [1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
]

START = (0, 0)
GOAL  = (7, 7)
ROWS, COLS = len(GRID), len(GRID[0])

def get_neighbors(pos):
    r, c = pos
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and GRID[nr][nc] == 0:
            yield (nr, nc)

def visualise_path(path, visited=None, title=''):
    grid_vis = np.array(GRID, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.cm.get_cmap('RdYlGn_r', 3)
    ax.imshow(grid_vis, cmap='Greys', vmin=0, vmax=1)

    if visited:
        for r, c in visited:
            ax.add_patch(mpatches.Rectangle((c-0.5, r-0.5), 1, 1,
                         color='lightblue', alpha=0.5))
    if path:
        for r, c in path:
            ax.add_patch(mpatches.Rectangle((c-0.5, r-0.5), 1, 1,
                         color='yellow', alpha=0.8))

    sr, sc = START
    gr, gc = GOAL
    ax.text(sc, sr, 'S', ha='center', va='center', fontsize=14, fontweight='bold', color='green')
    ax.text(gc, gr, 'G', ha='center', va='center', fontsize=14, fontweight='bold', color='red')
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

print("Grid world created. S=Start, G=Goal, black=wall")
visualise_path(None, title='Grid World')

## 2. Uninformed Search

Uninformed (blind) search has no information about how close a state is to the goal.

### Breadth-First Search (BFS)
- Explores all nodes at depth d before depth d+1
- Uses a **queue** (FIFO)
- **Complete** (finds a solution if one exists) and **optimal** (finds shortest path)
- Memory: O(b^d) — can be expensive

In [ ]:
def bfs(start, goal):
    queue    = deque([(start, [start])])
    visited  = {start}
    explored = []

    while queue:
        node, path = queue.popleft()
        explored.append(node)

        if node == goal:
            return path, explored

        for neighbor in get_neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None, explored

bfs_path, bfs_explored = bfs(START, GOAL)
print(f"BFS: path length={len(bfs_path)}, nodes explored={len(bfs_explored)}")
visualise_path(bfs_path, bfs_explored, f'BFS — Path length: {len(bfs_path)}, Explored: {len(bfs_explored)}')

In [ ]:
def dfs(start, goal):
    stack    = [(start, [start])]
    visited  = {start}
    explored = []

    while stack:
        node, path = stack.pop()
        explored.append(node)

        if node == goal:
            return path, explored

        for neighbor in get_neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                stack.append((neighbor, path + [neighbor]))

    return None, explored

dfs_path, dfs_explored = dfs(START, GOAL)
print(f"DFS: path length={len(dfs_path)}, nodes explored={len(dfs_explored)}")
visualise_path(dfs_path, dfs_explored, f'DFS — Path length: {len(dfs_path)}, Explored: {len(dfs_explored)}')

## 3. Informed Search — A*

A* uses a **heuristic function h(n)** to estimate the cost from node n to the goal.

$$f(n) = g(n) + h(n)$$

- **g(n):** actual cost from start to n
- **h(n):** estimated cost from n to goal (heuristic)
- **f(n):** estimated total cost through n

A* is **optimal** if h(n) is **admissible** (never overestimates the true cost).

Common admissible heuristics for grid worlds:
- **Manhattan distance:** |r1-r2| + |c1-c2| (for 4-directional movement)
- **Euclidean distance:** √((r1-r2)² + (c1-c2)²)

In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(start, goal, heuristic=manhattan):
    # Priority queue: (f, g, node, path)
    heap     = [(heuristic(start, goal), 0, start, [start])]
    visited  = {}
    explored = []

    while heap:
        f, g, node, path = heapq.heappop(heap)

        if node in visited:
            continue
        visited[node] = g
        explored.append(node)

        if node == goal:
            return path, explored

        for neighbor in get_neighbors(node):
            if neighbor not in visited:
                new_g = g + 1
                new_f = new_g + heuristic(neighbor, goal)
                heapq.heappush(heap, (new_f, new_g, neighbor, path + [neighbor]))

    return None, explored

astar_path, astar_explored = astar(START, GOAL)
print(f"A*:  path length={len(astar_path)}, nodes explored={len(astar_explored)}")
visualise_path(astar_path, astar_explored, f'A* — Path length: {len(astar_path)}, Explored: {len(astar_explored)}')

print(f"\nComparison:")
print(f"  BFS explored {len(bfs_explored)} nodes, path length {len(bfs_path)}")
print(f"  DFS explored {len(dfs_explored)} nodes, path length {len(dfs_path)}")
print(f"  A*  explored {len(astar_explored)} nodes, path length {len(astar_path)}")

## 4. Adversarial Search — Minimax

In two-player zero-sum games, one player tries to **maximise** their score while the other tries to **minimise** it.

**Minimax** explores the full game tree and picks the move that maximises the minimum gain (assuming the opponent plays optimally).

**Alpha-Beta Pruning** cuts branches that cannot affect the final decision — same result as minimax but much faster.

In [ ]:
# ── Tic-Tac-Toe with Minimax + Alpha-Beta ──────────────────────────────────

def check_winner(board):
    lines = [
        [0,1,2],[3,4,5],[6,7,8],  # rows
        [0,3,6],[1,4,7],[2,5,8],  # cols
        [0,4,8],[2,4,6]           # diagonals
    ]
    for line in lines:
        vals = [board[i] for i in line]
        if vals == ['X','X','X']: return 'X'
        if vals == ['O','O','O']: return 'O'
    return None

def minimax(board, depth, is_max, alpha=-math.inf, beta=math.inf):
    winner = check_winner(board)
    if winner == 'X': return 10 - depth
    if winner == 'O': return depth - 10
    if ' ' not in board: return 0  # draw

    if is_max:  # X's turn (maximiser)
        best = -math.inf
        for i in range(9):
            if board[i] == ' ':
                board[i] = 'X'
                best = max(best, minimax(board, depth+1, False, alpha, beta))
                board[i] = ' '
                alpha = max(alpha, best)
                if beta <= alpha: break  # β cutoff
        return best
    else:       # O's turn (minimiser)
        best = math.inf
        for i in range(9):
            if board[i] == ' ':
                board[i] = 'O'
                best = min(best, minimax(board, depth+1, True, alpha, beta))
                board[i] = ' '
                beta = min(beta, best)
                if beta <= alpha: break  # α cutoff
        return best

def best_move(board):
    best_val, best_idx = -math.inf, -1
    for i in range(9):
        if board[i] == ' ':
            board[i] = 'X'
            val = minimax(board, 0, False)
            board[i] = ' '
            if val > best_val:
                best_val, best_idx = val, i
    return best_idx

def print_board(board):
    for i in range(0, 9, 3):
        print(f" {board[i]} | {board[i+1]} | {board[i+2]}")
        if i < 6: print('---+---+---')

# Demo: AI plays as X from a mid-game position
board = ['O', ' ', ' ',
         ' ', 'X', ' ',
         ' ', ' ', 'O']

print("Current board:")
print_board(board)
move = best_move(board)
print(f"\nMinimax (X) plays position {move}")
board[move] = 'X'
print_board(board)

## 5. Monte Carlo Tree Search (MCTS)

MCTS is the algorithm that made AlphaGo possible. Unlike minimax, it doesn't need a hand-crafted evaluation function — it uses **random simulations** (rollouts) to estimate the value of positions.

**Four phases:**
1. **Selection:** traverse the tree using UCB1 to balance exploration/exploitation
2. **Expansion:** add a new child node
3. **Simulation:** play out a random game from the new node
4. **Backpropagation:** update win/visit counts up the tree

$$UCB1 = \frac{w_i}{n_i} + C\sqrt{\frac{\ln N}{n_i}}$$

- w_i: wins from node i
- n_i: visits to node i  
- N: total visits to parent
- C: exploration constant (typically √2)

In [ ]:
class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state    = state[:]
        self.parent   = parent
        self.move     = move
        self.children = []
        self.wins     = 0
        self.visits   = 0
        self.untried  = [i for i in range(9) if state[i] == ' ']

    def ucb1(self, C=1.414):
        if self.visits == 0:
            return float('inf')
        return self.wins/self.visits + C * math.sqrt(math.log(self.parent.visits)/self.visits)

    def best_child(self):
        return max(self.children, key=lambda c: c.ucb1())


def mcts_rollout(state, player):
    """Random playout from state. Returns 1 if player wins, -1 if loses, 0 draw."""
    board = state[:]
    current = player
    for _ in range(9):
        empty = [i for i in range(9) if board[i] == ' ']
        if not empty: break
        board[random.choice(empty)] = current
        winner = check_winner(board)
        if winner:
            return 1 if winner == player else -1
        current = 'O' if current == 'X' else 'X'
    return 0


def mcts(root_state, player='X', n_simulations=500):
    root = MCTSNode(root_state)

    for _ in range(n_simulations):
        node = root
        state = root_state[:]
        current_player = player

        # 1. Selection
        while not node.untried and node.children:
            node = node.best_child()
            state[node.move] = current_player
            current_player = 'O' if current_player == 'X' else 'X'

        # 2. Expansion
        if node.untried:
            move = random.choice(node.untried)
            node.untried.remove(move)
            state[move] = current_player
            child = MCTSNode(state, parent=node, move=move)
            node.children.append(child)
            node = child
            current_player = 'O' if current_player == 'X' else 'X'

        # 3. Simulation
        result = mcts_rollout(state, player)

        # 4. Backpropagation
        while node:
            node.visits += 1
            node.wins   += result
            node = node.parent

    # Return move with most visits
    return max(root.children, key=lambda c: c.visits).move


# Demo MCTS
board = ['O', ' ', ' ',
         ' ', 'X', ' ',
         ' ', ' ', 'O']

print("Current board:")
print_board(board)

mcts_move = mcts(board, player='X', n_simulations=1000)
print(f"\nMCTS (X, 1000 simulations) plays position {mcts_move}")
board[mcts_move] = 'X'
print_board(board)

## 6. Algorithm Comparison

| Algorithm | Complete? | Optimal? | Time | Space | Use Case |
|---|---|---|---|---|---|
| BFS | ✅ | ✅ (unit cost) | O(b^d) | O(b^d) | Shortest path, small graphs |
| DFS | ✅ | ❌ | O(b^m) | O(bm) | Memory-constrained, deep solutions |
| Greedy | ❌ | ❌ | O(b^m) | O(b^m) | Fast but suboptimal |
| A* | ✅ | ✅ (admissible h) | O(b^d) | O(b^d) | Pathfinding, planning |
| Minimax | ✅ | ✅ | O(b^m) | O(bm) | Two-player games |
| MCTS | ✅ | ✅ (∞ sims) | O(n_sims) | O(n_sims) | Complex games (Go, Chess) |

b = branching factor, d = solution depth, m = max depth

## 7. Summary

### Key Takeaways
- **BFS** finds the shortest path but uses lots of memory
- **DFS** uses less memory but may find suboptimal paths
- **A*** is optimal and efficient when given an admissible heuristic
- **Minimax** with alpha-beta pruning is the foundation of game-playing AI
- **MCTS** uses random simulations to evaluate positions — no hand-crafted evaluation needed
- AlphaGo combined MCTS with deep neural networks for superhuman Go play

### Next Steps
- **I15a** — Reinforcement Learning (learning to search via rewards)
- **E10** — Deep Reinforcement Learning (DQN, AlphaGo Zero)